## EDA
1. 데이터 구조 확인 
  - 데이터 크기 확인(shape 함수)
  - 컬럼명, 타입 확인

In [ ]:
# Y(종속변수): YOUTH_NET_MOVE_RATE (청년순이동률)

# 임포트
import pandas as pd
from IPython.core.interactiveshell import InteractiveShell 

# print 함수 모두 출력
InteractiveShell.ast_node_interactivity = "all"

# 모든 행과 열 보기
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# 데이터 불러오기
df = pd.read_csv("../../data/02-1_EDA/data_add.csv", encoding="utf-8-sig")
col_mapping = pd.read_csv("../../data/02-1_EDA/col-mapping.csv", encoding="utf-8-sig")
df_type = df.dtypes
# 데이터 크기 확인 
print(f"행 수: {df.shape[0]}, 열 수: {df.shape[1]}")
print(df_type)

In [ ]:
# SGG_CODE, YEAR, SGG_NAME 컬럼 제외한 모든 열이 수치형 변수
# df_type 데이터 타입과 col_mapping 데이터 타입 일치하는지 여부 비교
# 1. 실제 df의 dtype 추출
df_type = df.dtypes.astype(str).reset_index()
df_type.columns = ["컬럼명", "실제_dtype"]

# 2. col_mapping에서 정의된 dtype 변환
dtype_map = {"int": "int64", "float": "float64", "object": "object", "float64": "float64", "int64": "int64"}
col_mapping["정의_dtype"] = col_mapping["데이터타입"].map(dtype_map)

# 3. 병합
compare = pd.merge(df_type, col_mapping[["컬럼명", "정의_dtype"]], on="컬럼명", how="inner")

# 4. 일치 여부
compare["일치여부"] = ( compare["실제_dtype"] == compare["정의_dtype"])

# 일치여부가 false인 행 출력
# print(compare[compare["일치여부"] == False])
if len(compare[compare["일치여부"] == False]) == 0:
    print("모든 컬럼의 데이터 타입이 일치합니다.")
else:
    print(compare[compare["일치여부"] == False])

In [ ]:
# 출력 결과 WELFARE_EXPENSE, WATER_SUPPLY_RATE, SEWERAGE_SUPPLY_RATE 데이터 타입 일치하지 않음
# 확인 결과
# WELFARE_EXPENSE: 정수로 변경 (비용이므로)
# WATER_SUPPLY_RATE, SEWERAGE_SUPPLY_RATE: 실수 맞음

# df["WELFARE_EXPENSE"] = df["WELFARE_EXPENSE"].astype(int)

# 데이터 타입 확인
# print(df["WELFARE_EXPENSE"].dtypes)

In [ ]:
# Y(종속변수): YOUTH_NET_MOVE_RATE (청년순이동률)
# 결측치 처리는 이미 진행함
# 이상치 탐지

# 단변량 이상치 탐지
# 1. Z-score

# 임포트
from scipy import stats
import numpy as np

# SGG_NAME, SGG_CODE, YEAR, NET_MOVE, Y 컬럼 제외
df_col = df.drop(columns=["SGG_NAME", "SGG_CODE", "YEAR", "NET_MOVE", "Y"])

# 결과 저장 리스트
outlier_results_zscore = []

for col in df_col.columns:
    z = np.abs(stats.zscore(df_col[col]))
    outliers = df_col[z > 3]   # 이상치 행 추출
    
    outlier_results_zscore.append({
        "변수명": col,
        "Z-score이상치 수": len(outliers),
        "Z-score이상치 비율(%)": round(len(outliers) / len(df_col) * 100, 2)
    })

# 데이터프레임 변환
outlier_df = pd.DataFrame(outlier_results_zscore)
print(outlier_df)

In [ ]:
# IQR (4분위수 범위)
outlier_results_iqr = []

for col in df_col.columns:
    Q1 = df_col[col].quantile(0.25)
    Q3 = df_col[col].quantile(0.75)
    IQR = Q3 - Q1

    outliers = df_col[(df_col[col] < Q1 - 1.5*IQR) | (df_col[col] > Q3 + 1.5*IQR)]
    
    outlier_results_iqr.append({
        "변수명": col,
        "IQR이상치 수": len(outliers),
        "IQR이상치 비율(%)": round(len(outliers) / len(df_col) * 100, 2)
    })

# 데이터프레임 변환
outlier_df_iqr = pd.DataFrame(outlier_results_iqr)

# 비율 높은 순 정렬
outlier_df_iqr = outlier_df_iqr.sort_values(by="IQR이상치 비율(%)", ascending=False)

print(outlier_df_iqr)


In [ ]:
# 두 결과 병합 (변수명 기준)
outlier_compare = pd.merge(
    outlier_df, outlier_df_iqr, 
    on="변수명", how="inner"
)

# 비율 높은 순으로 정렬
outlier_compare = outlier_compare.sort_values(
    by=["Z-score이상치 비율(%)", "IQR이상치 비율(%)"], 
    ascending=False
)

outlier_compare.to_csv("../../data/02-1_EDA/outlier_compare.csv", index=False, encoding="utf-8-sig")

print(outlier_compare)

In [ ]:
# boxplot(박스플롯)
# 데이터 분포와 이상치 시각화

# 한글 설정
import matplotlib.pyplot as plt
from matplotlib import rcParams

# 한글 폰트 설정
rcParams['font.family'] = 'Malgun Gothic'
rcParams['axes.unicode_minus'] = False  # 마이너스 깨짐 방지


# 임포트
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

for col in df_col.columns:
    ax1 = plt.figure(figsize=(10, 6))
    ax2 = sns.boxplot(x=df_col[col])
    ax3 = plt.title(f"{col} 분포")
    plt.savefig(f"../../data/02-1_EDA/boxplot/{col}_boxplot.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
# 히스토그램 & KDE-PLOT

for col in df_col.columns:
    ax1 = plt.figure(figsize=(10, 6))
    ax2 = sns.histplot(df_col[col], bins=20, kde=True)  # kde=True 하면 밀도곡선도 같이 나옴
    ax3 = plt.title(f"{col} 분포")
    plt.savefig(f"../../data/02-1_EDA/hist/{col}_hist_kde.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

3. 기본 통계 분석
  - 평균, 중앙값, 최솟값, 최댓값, 기본 통계
  - 수치형 변수: 분포 확인 (`hist()`, `plot.kde()` 함수 사용)

In [ ]:
# describe() 요약 통계량을 변수별로 정리
summary_stats = df_col.describe().T.reset_index()

# 컬럼명 변경
summary_stats = summary_stats.rename(columns={
    "index": "변수명",
    "count": "개수",
    "mean": "평균",
    "std": "표준편차",
    "min": "최솟값",
    "25%": "Q1(25%)",
    "50%": "중앙값(50%)",
    "75%": "Q3(75%)",
    "max": "최댓값"
})

# 두 결과 병합 (변수명 기준)
outlier_compare_df = pd.merge(
    outlier_compare, summary_stats, 
    on="변수명", how="inner"
)
outlier_compare_df.to_csv("../../data/02-1_EDA/outlier_compare_df.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import shapiro, normaltest, anderson, skew, kurtosis, probplot
import matplotlib.pyplot as plt

def check_normality_series(s: pd.Series, alpha: float = 0.05):
    """
    단일 Series 정규성 검정 요약 반환.
    반환: dict (샘플수, 왜도, 첨도, Shapiro p, D'Agostino p, AD 통계량/임계값, 판정)
    """
    s = s.dropna().astype(float)
    n = len(s)
    if n < 8:  # scipy 권장 최소 샘플
        return {
            "n": n, "skew": np.nan, "kurtosis": np.nan,
            "shapiro_p": np.nan, "dagostino_p": np.nan,
            "anderson_stat": np.nan, "anderson_crit@5%": np.nan,
            "normal@alpha=0.05": "insufficient n (<8)"
        }

    # 요약 통계
    sk = skew(s)
    kt = kurtosis(s, fisher=True)  # 0이면 정규분포 첨도

    # Shapiro–Wilk
    try:
        sh_p = shapiro(s)[1]
    except Exception:
        sh_p = np.nan

    # D’Agostino K^2
    try:
        dag_p = normaltest(s)[1]
    except Exception:
        dag_p = np.nan

    # Anderson–Darling (정규분포 가정)
    ad = anderson(s, dist='norm')
    ad_stat = ad.statistic
    # 임계값은 [15%, 10%, 5%, 2.5%, 1%] 순서
    # 5% 임계값 사용
    try:
        crit_5 = ad.critical_values[list(ad.significance_level).index(5.0)]
    except Exception:
        # 보통 [15,10,5,2.5,1] 고정이므로 인덱스 2
        crit_5 = ad.critical_values[2]

    # 판정: p>=alpha 이면 채택(정규), AD는 stat <= crit 이면 정규
    votes = []
    if not np.isnan(sh_p):  votes.append(sh_p >= alpha)
    if not np.isnan(dag_p): votes.append(dag_p >= alpha)
    votes.append(ad_stat <= crit_5)

    if len(votes) == 0:
        verdict = "undetermined"
    else:
        verdict = "normal" if sum(votes) >= (len(votes) / 2) else "non-normal"

    return {
        "n": n,
        "skew": sk,
        "kurtosis": kt,
        "shapiro_p": sh_p,
        "dagostino_p": dag_p,
        "anderson_stat": ad_stat,
        "anderson_crit@5%": crit_5,
        "normal@alpha=0.05": verdict
    }


def normality_summary(df: pd.DataFrame, cols=None, alpha: float = 0.05):
    """
    여러 컬럼에 대해 정규성 검정 요약 DataFrame 반환.
    """
    if cols is None:
        # 숫자형만
        cols = df.select_dtypes(include=[np.number]).columns.tolist()

    rows = []
    for c in cols:
        res = check_normality_series(df[c], alpha=alpha)
        rows.append({"변수명": c, **res})

    out = pd.DataFrame(rows)
    # 해석에 도움 되도록 정렬: 비정규 먼저, 그 다음 샘플수↓
    order = pd.CategoricalDtype(categories=["non-normal","normal","undetermined","insufficient n (<8)"], ordered=True)
    out["normal@alpha=0.05"] = out["normal@alpha=0.05"].astype(order)
    out = out.sort_values(["normal@alpha=0.05", "n"], ascending=[True, False]).reset_index(drop=True)
    return out


def qqplot(series: pd.Series, title: str = None):
    """
    간단 Q-Q plot (정규분포 기준).
    """
    plt.figure(figsize=(6,6))
    probplot(series.dropna().astype(float), dist="norm", plot=plt)
    if title:
        plt.title(title)
    plt.show()


In [ ]:
# 예: df_col 전체 요약표
summary = normality_summary(df_col, alpha=0.05)
summary.to_csv("../../data/02-1_EDA/summary_stats.csv", index=False, encoding="utf-8-sig")
print(summary.head())

# 특정 변수 Q-Q plot
qqplot(df_col["MIN_TEMP"], title="MIN_TEMP Q-Q Plot")
qqplot(df_col["MIN_TEMP"], title=" Q-Q Plot")

In [ ]:
## clip은 진행하지 않고 log변환만 진행하기

# 불러오기
file_outlier_new = "../../data/02-1_EDA/outlier_test_result_with_recommendations.csv"
df_outlier_new = pd.read_csv(file_outlier_new)

# log 또는 log_clip이 추천 처리 방법인 변수만 추출
df_log_candidates = df_outlier_new[
    df_outlier_new["추천 처리 방법"].str.contains("log", case=False, na=False)
][["변수명", "추천 처리 방법", "skew", "kurtosis", "normal@alpha=0.05"]]

df_log_candidates

In [ ]:
# 15개 변수 로그변환 진행

# 직전에 찾은 outlier 파일에서 log/log_clip 변수 추출
file_outlier_new = "../../data/02-1_EDA/outlier_test_result_with_recommendations.csv"
df_outlier_new = pd.read_csv(file_outlier_new)

# log / log_clip 권장된 변수 목록 (총 15개일 것으로 가정)
log_vars = df_outlier_new[
    df_outlier_new["추천 처리 방법"].str.contains("log", case=False, na=False)
]["변수명"].tolist()

# 로그 변환 적용 (log1p)
df_col = df_col.copy()
for var in log_vars:
    if var in df_col.columns:
        df_col[var] = np.log1p(df_col[var])

# 결과 저장
output_path_data_add_log = "./result/data_add_log.csv"
df_col.to_csv(output_path_data_add_log, index=False, encoding="utf-8-sig")

output_path_data_add_log

In [ ]:
# 히스토그램, 밀도 그래프(KDE)

for col in df_col.columns:
    ax1 = plt.figure(figsize=(10, 6))
    ax2 = sns.histplot(df_col[col], bins=20, kde=True)  # kde=True 하면 밀도곡선도 같이 나옴
    ax3 = plt.title(f"{col} 분포 (히스토그램)")
    plt.show()

4. 변수들의 관계 탐색 
  - 상관계수: 0.8 이상 다중공선성 의심
  - 산점도

In [ ]:
### 코드 계속 같기때문에 함수화 진행

# 상관계수 히트맵 함수
def corr_hitmap(
    df_col: pd.DataFrame, 
    col_list: list = None,
    cofog_name: str = None
    ) -> pd.DataFrame:
    
    # 상관계수 행렬
    if col_list is None:
        corr = df_col.corr()
    else:
        corr = df_col[col_list].corr()

    # 히트맵 시각화
    if col_list is None:
        ax1 = plt.figure(figsize=(100,80))
    else:
        ax1 = plt.figure(figsize=(20, 16))

    ax2 = sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
    if cofog_name is None:
        ax3 = plt.title(" 전체 상관계수")
    else:
        ax3 = plt.title(f"{cofog_name} 상관계수")

    plt.show()
    
    return corr

In [ ]:
def high_corr_pairs(
    corr: pd.DataFrame, 
    ) -> pd.DataFrame:
    # 절대값 기준 0.8 이상인 쌍 찾기
    high_corr = corr[(corr.abs() >= 0.8) & (corr.abs() < 1.0)]

    # stack으로 펼치기
    corr_pairs = high_corr.stack().reset_index()
    corr_pairs.columns = ['VAR1', 'VAR2', 'CORR']

    # 중복 제거
    corr_pairs = corr_pairs[corr_pairs['VAR1'] < corr_pairs['VAR2']]
    
    print(corr_pairs)
    return corr_pairs

In [ ]:
# 전체 상관계수 히트맵
corr_total = corr_hitmap(df_col)

In [ ]:
# 전체 상관계수 값(0.8 이상) 높은 변수 쌍
corr_total_list = high_corr_pairs(corr_total)

In [ ]:
# cofog별 상관계수 확인
# 기후: climate
# 경제활동: economic_affairs
# 교육: education
# 환경보호: environment_protection
# 보건: health
# 주거및지역사회건설: housing_community
# 일반공공행정: public_administration
# 공공질서및안전: public_safety
# 휴양및문화: recreation_culture
# 사회보호: social_welfare

col_cf = pd.read_csv("../../data/02-1_EDA/col-classification.csv", encoding="utf-8-sig")
public_administration = col_cf.loc[col_cf["classification"] == "일반공공행정", "col_name"].tolist()
climate = col_cf.loc[col_cf["classification"] == "기후", "col_name"].tolist()
economic_affairs = col_cf.loc[col_cf["classification"] == "경제활동", "col_name"].tolist()
education = col_cf.loc[col_cf["classification"] == "교육", "col_name"].tolist()
environment_protection = col_cf.loc[col_cf["classification"] == "환경보호", "col_name"].tolist()
health = col_cf.loc[col_cf["classification"] == "보건", "col_name"].tolist()
housing_community = col_cf.loc[col_cf["classification"] == "주거및지역사회건설", "col_name"].tolist()
public_safety = col_cf.loc[col_cf["classification"] == "공공질서및안전", "col_name"].tolist()
recreation_culture = col_cf.loc[col_cf["classification"] == "휴양및문화", "col_name"].tolist()
social_welfare = col_cf.loc[col_cf["classification"] == "사회보호", "col_name"].tolist()

In [ ]:
# 일반공공행정 상관계수 히트맵(Y(YOUTH_NET_MOVE_RATE) 제외)
# public_administration.remove("YOUTH_NET_MOVE_RATE")
# public_administration.remove("YOUTH_NET_MOVE")
public_administration.remove("NET_MOVE")

corr_PA = corr_hitmap(df_col, public_administration, "일반공공행정")

In [ ]:
# 일반공공행정 상관계수 값(0.8 이상) 높은 변수 쌍
corr_PA_list = high_corr_pairs(corr_PA)

In [ ]:
# 주거및지역사회건설 상관계수 히트맵
corr_HC = corr_hitmap(df_col, housing_community, "주거및지역사회건설")

In [ ]:
# 주거및지역사회건설 상관계수 값(0.8 이상) 높은 변수 쌍
corr_HC_list = high_corr_pairs(corr_HC)

In [ ]:
# 보건 상관계수 히트맵
corr_H = corr_hitmap(df_col, health, "보건")

In [ ]:
# 보건 상관계수 값(0.8 이상) 높은 변수 쌍
corr_H_list = high_corr_pairs(corr_H)

In [ ]:
# 휴양및문화 상관계수 히트맵
corr_RC = corr_hitmap(df_col, recreation_culture, "휴양및문화")

In [ ]:
# 휴양및문화 상관계수 값(0.8 이상) 높은 변수 쌍
corr_RC_list = high_corr_pairs(corr_RC) 

In [ ]:
# 교육 상관계수 상관계수 히트맵
corr_E = corr_hitmap(df_col, education, "교육")

In [ ]:
# 교육 상관계수 값(0.8 이상) 높은 변수 쌍
corr_E_list = high_corr_pairs(corr_E)

In [ ]:
# 공공질서및안전 상관계수 히트맵
corr_PS = corr_hitmap(df_col, public_safety, "공공질서및안전")

In [ ]:
# 공공질서및안전 상관계수 값(0.8 이상) 높은 변수 쌍
corr_PS_list = high_corr_pairs(corr_PS)

In [ ]:
# 경제활동 상관계수 히트맵
corr_EA = corr_hitmap(df_col, economic_affairs, "경제활동")

In [ ]:
# 경제활동 상관계수 값(0.8 이상) 높은 변수 쌍
corr_EA_list = high_corr_pairs(corr_EA)

In [ ]:
# 사회보호 상관계수 히트맵

corr_SW = corr_hitmap(df_col, social_welfare, "사회보호")

In [ ]:
# 사회보호 상관계수 값(0.8 이상) 높은 변수 쌍
corr_SW_list = high_corr_pairs(corr_SW)

In [ ]:
# 환경보호 상관계수 히트맵
corr_EP = corr_hitmap(df_col, environment_protection, "환경보호")

In [ ]:
# 환경보호 상관계수 값(0.8 이상) 높은 변수 쌍
corr_EP_list = high_corr_pairs(corr_EP)

In [ ]:
# 기후 상관계수 히트맵
corr_C = corr_hitmap(df_col, climate, "기후")

In [ ]:
# 기후 상관계수 값(0.8 이상) 높은 변수 쌍
corr_C_list = high_corr_pairs(corr_C)

In [ ]:
# 산점도
import seaborn as sns
import matplotlib.pyplot as plt

# df: 데이터프레임
# cols: 산점도를 보고 싶은 컬럼 리스트
def scatter_matrix(
    df: pd.DataFrame, 
    cols: list =None
    ) -> None:
    if cols is None:  # 지정 안 하면 전체 수치형 변수
        cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
    
    sns.pairplot(df[cols], diag_kind="kde", plot_kws={"alpha":0.5, "s":10})
    plt.show()

In [ ]:
# 일반공공행정 산점도
scatter_matrix(df_col, public_administration)

In [ ]:
# 주거및지역사회건설 산점도
scatter_matrix(df_col, housing_community)

In [ ]:
# 보건 산점도
scatter_matrix(df_col, health)

In [ ]:
# 휴양및문화 산점도
scatter_matrix(df_col, recreation_culture)

In [ ]:
# 교육 산점도
scatter_matrix(df_col, education)

In [ ]:
# 공공질서및안전 산점도
scatter_matrix(df_col, public_safety)

In [ ]:
# 경제활동 산점도
scatter_matrix(df_col, economic_affairs)

In [ ]:
# 사회보호 산점도
scatter_matrix(df_col, social_welfare)

In [ ]:
# 환경보호 산점도
scatter_matrix(df_col, environment_protection)

In [ ]:
# 기후 산점도
scatter_matrix(df_col, climate)

5. 다중공선성(VIF) 확인
  - 전체 다른 변수들과의 다중 선형적 관계
  - 분산 팽창 지수
  - 결정계수 R^2 이용
  - VIF = 1/(1-R^2)
  - VIF > 10일 때: 심각한 다중공선성
  - VIF > 5이 때: 주의 필요

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

def calculate_vif(
    df: pd.DataFrame, 
    col_list: list =None
    ) -> pd.DataFrame:
    
    # 사용할 컬럼 선택
    if col_list is None:
        col_list = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
        X = add_constant(df)
    else:
        X = add_constant(df[col_list])  # 상수항 추가
    
    vif_data = pd.DataFrame()
    vif_data["변수명"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i)
                       for i in range(X.shape[1])]

    # inf → NaN 처리
    vif_data["VIF"] = vif_data["VIF"].replace([np.inf, -np.inf], np.nan)

    # 소수점 둘째자리까지 반올림
    vif_data["VIF"] = vif_data["VIF"].round(2)

    # NaN은 다시 "inf"로 표현 (표시용)
    vif_data["VIF"] = vif_data["VIF"].fillna("inf")
        
    return vif_data


In [ ]:
# 전체 다중공선성
# df_col = df.drop(columns=["YOUTH_NET_MOVE", "YOUTH_NET_MOVE_RATE"], axis=1)

vif_total = calculate_vif(df_col)
vif_total.sort_values(by="VIF", ascending=False)

In [ ]:
# 일반공공행정 산점도
vif_PA = calculate_vif(df_col, public_administration)
print(vif_PA)

In [ ]:
# 주거및지역사회건설
vif_HC = calculate_vif(df_col, housing_community)
print(vif_HC)

In [ ]:
# 보건
vif_H = calculate_vif(df_col, health)
print(vif_H)

In [ ]:
# 휴양및문화
vif_RC = calculate_vif(df_col, recreation_culture)
print(vif_RC)

In [ ]:
# 교육
vif_E = calculate_vif(df_col, education)
print(vif_E)

In [ ]:
# 공공질서및안전
vif_PS = calculate_vif(df_col, public_safety)
print(vif_PS)

In [ ]:
# 경제활동
vif_EA = calculate_vif(df_col, economic_affairs)
print(vif_EA)

In [ ]:
# 사회보호
vif_SW = calculate_vif(df_col, social_welfare)
print(vif_SW)

In [ ]:
# 환경보호
vif_EP = calculate_vif(df_col, environment_protection)
print(vif_EP)

In [ ]:
# 기후
vif_C = calculate_vif(df_col, climate)
print(vif_C)

In [ ]:
df_col.to_csv("../../data/02-1_EDA/reduction_data.csv", index=False, encoding="utf-8-sig")